# Liquidation Risk Analysis

Analyzes worst-case simultaneous unrealized loss across concurrent open positions from backtest data. Used to size capital requirements in `THESIS.md`.

**Methodology**: For each hour in the backtest, mark every open position to market using actual hourly prices, sum their unrealized P&L, and track the worst moment.

**Not**: the sum of each position's lifetime worst MAE (that's pessimistic — assumes all hit their individual worst simultaneously).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

NOTIONAL = 50000
REPORTS_DIR = Path('../data/reports')
CANDLES_DIR = Path('../data/candles')
COINS = ['BTC', 'ETH', 'SOL', 'AVAX', 'DOGE', 'LINK']

## Load data

In [ ]:
# Load backtest trades (run verification/step1_backtest.py first if missing)
bt = pd.read_csv(REPORTS_DIR / 'backtest_trades.csv')
bt['entry_dt'] = pd.to_datetime(bt['entry_ts'], unit='ms', utc=True)
bt['exit_dt'] = pd.to_datetime(bt['exit_ts'], unit='ms', utc=True)

# Load hourly candles for all coins
prices = {}
for coin in COINS:
    df = pd.read_csv(CANDLES_DIR / f'{coin}_1h.csv')
    df['dt'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.set_index('dt').sort_index()
    df = df[~df.index.duplicated(keep='first')]
    prices[coin] = df['close']

prices = pd.DataFrame(prices).ffill().dropna()
print(f'Loaded {len(bt)} trades and {len(prices)} hourly price bars')

## Compute simultaneous unrealized P&L hour by hour

In [ ]:
def position_unrealized(trade, current_a, current_b):
    """Unrealized P&L for a single position at current prices."""
    ea, eb = trade['entry_price_a'], trade['entry_price_b']
    if trade['direction'] == 'long_ratio':
        return NOTIONAL * (current_a - ea) / ea + NOTIONAL * (eb - current_b) / eb
    else:
        return NOTIONAL * (ea - current_a) / ea + NOTIONAL * (current_b - eb) / eb

concurrent = []
for hour in prices.index:
    active = bt[(bt['entry_dt'] <= hour) & (bt['exit_dt'] > hour)]
    if len(active) == 0:
        concurrent.append({'hour': hour, 'n_positions': 0, 'unrealized': 0})
        continue

    total = 0
    for _, t in active.iterrows():
        coin_a, coin_b = t['pair'].split('/')
        total += position_unrealized(t, prices.loc[hour, coin_a], prices.loc[hour, coin_b])

    concurrent.append({'hour': hour, 'n_positions': len(active), 'unrealized': total})

conc_df = pd.DataFrame(concurrent)
print(f'Computed unrealized for {len(conc_df)} hours')

## Distribution summary

In [ ]:
with_pos = conc_df[conc_df['n_positions'] > 0]

print('Distribution of simultaneous unrealized P&L:')
print(f'  Worst:    ${with_pos["unrealized"].min():+,.0f}')
print(f'  5th %ile: ${with_pos["unrealized"].quantile(0.05):+,.0f}')
print(f'  Median:   ${with_pos["unrealized"].median():+,.0f}')
print(f'  95th %ile:${with_pos["unrealized"].quantile(0.95):+,.0f}')
print(f'  Best:     ${with_pos["unrealized"].max():+,.0f}')
print()

print('Time spent at each concurrent position count:')
for n in sorted(conc_df['n_positions'].unique()):
    pct = (conc_df['n_positions'] == n).sum() / len(conc_df) * 100
    print(f'  {n} positions: {pct:.1f}% of hours')

## Worst moment — full breakdown

In [ ]:
worst_idx = conc_df['unrealized'].idxmin()
worst_hour = conc_df.loc[worst_idx, 'hour']
worst_total = conc_df.loc[worst_idx, 'unrealized']

print(f'Worst moment: {worst_hour}')
print(f'Total simultaneous unrealized: ${worst_total:+,.0f}')
print()

print('Spot prices at that moment:')
for coin in COINS:
    print(f'  {coin:>6}: ${prices.loc[worst_hour, coin]:,.4f}')
print()

active = bt[(bt['entry_dt'] <= worst_hour) & (bt['exit_dt'] > worst_hour)].copy()
active = active.sort_values('entry_dt')

for _, t in active.iterrows():
    coin_a, coin_b = t['pair'].split('/')
    cur_a = prices.loc[worst_hour, coin_a]
    cur_b = prices.loc[worst_hour, coin_b]
    ea, eb = t['entry_price_a'], t['entry_price_b']

    if t['direction'] == 'long_ratio':
        pnl_a = NOTIONAL * (cur_a - ea) / ea
        pnl_b = NOTIONAL * (eb - cur_b) / eb
        dir_desc = f'LONG {coin_a} / SHORT {coin_b}'
    else:
        pnl_a = NOTIONAL * (ea - cur_a) / ea
        pnl_b = NOTIONAL * (cur_b - eb) / eb
        dir_desc = f'SHORT {coin_a} / LONG {coin_b}'

    total = pnl_a + pnl_b
    hours_held = (worst_hour - t['entry_dt']).total_seconds() / 3600

    print(f'  {t["pair"]} — {dir_desc}')
    print(f'    Entry: {t["entry_dt"]} at z={t["entry_z"]:+.2f}')
    print(f'    Held: {hours_held:.0f}h (full hold: {t["hours_held"]}h, exit at {t["exit_dt"]})')
    print(f'    {coin_a}: ${ea:,.4f} → ${cur_a:,.4f} ({(cur_a-ea)/ea*100:+.2f}%) → leg P&L: ${pnl_a:+,.0f}')
    print(f'    {coin_b}: ${eb:,.4f} → ${cur_b:,.4f} ({(cur_b-eb)/eb*100:+.2f}%) → leg P&L: ${pnl_b:+,.0f}')
    print(f'    Position unrealized at worst: ${total:+,.0f}')
    print(f'    Final outcome: ${t["net_pnl"]:+,.0f} (exit z={t["exit_z"]:+.2f}, {t["exit_reason"]})')
    print()

## Timeline around worst moment (±6 hours)

In [ ]:
window_start = worst_hour - pd.Timedelta(hours=6)
window_end = worst_hour + pd.Timedelta(hours=12)
window = conc_df[(conc_df['hour'] >= window_start) & (conc_df['hour'] <= window_end)].copy()

print(f'{"Hour":>20} {"N Pos":>7} {"Unrealized":>15}')
print(f'{"-"*20} {"-"*7} {"-"*15}')
for _, row in window.iterrows():
    marker = '  ← WORST' if row['hour'] == worst_hour else ''
    print(f'{row["hour"].strftime("%Y-%m-%d %H:%M"):>20} {row["n_positions"]:>7} ${row["unrealized"]:>+13,.0f}{marker}')

## Capital requirements by leverage

In [ ]:
worst_case = abs(worst_total)
max_notional = 4 * 2 * NOTIONAL  # 4 pairs x 2 legs x $50K

print(f'Max notional (4 pairs, 2 legs, ${NOTIONAL:,}/leg): ${max_notional:,}')
print(f'Worst-case simultaneous unrealized: ${worst_case:,}')
print()
print(f'{"Leverage":>9} {"Margin":>10} {"Capital ($120K safety)":>25} {"Buffer":>10} {"Cushion vs worst":>18}')
print('-' * 80)

for lev in [1, 2, 3, 5, 7, 10, 15, 20]:
    margin = max_notional / lev
    # For the recommended capital, add ~$40K-$50K buffer
    # Keep buffer as a multiple of worst case
    if lev <= 3:
        buffer = max(50000, worst_case * 2)
    elif lev <= 5:
        buffer = 40000
    else:
        buffer = 20000
    capital = margin + buffer
    cushion = buffer / worst_case
    verdict = '✅' if cushion >= 1.5 else '⚠️' if cushion >= 1.0 else '❌'
    print(f'{lev:>7}x  ${margin:>8,.0f} ${capital:>23,.0f} ${buffer:>8,.0f} {cushion:>14.1f}x  {verdict}')

print()
print('Cushion = buffer / worst-case simultaneous unrealized')
print('Recommendation: 5x leverage with $120K capital (3x cushion)')

## Export results

In [ ]:
# Save the concurrent unrealized time series for later inspection
out_path = REPORTS_DIR / 'concurrent_unrealized.csv'
conc_df.to_csv(out_path, index=False)
print(f'Saved hour-by-hour concurrent unrealized to {out_path}')